# Day 13: Capstone Project — Build Your Own Production Agent 🏆

**Agentic AI Hands-On Course** | Dr. Kanthi Kiran Sirra | Sr. AI Engineer

**Student:** Nishant Sahoo


## Capstone Plan

**Name:** Nishant Sahoo

**Domain:** Research Paper Q&A — Deep Reinforcement Learning (DRL)

**User:** PhD students and AI researchers.

**Success looks like:** The agent answers domain-specific questions with faithfulness ≥ 0.7, correctly declines out-of-scope questions, and maintains conversation memory across turns within a session.

**Tools used:**
- **PyMuPDF (`fitz`)** — PDF text extraction and chunking from research papers
- **ChromaDB** — in-memory vector store for storing and querying document chunks
- **SentenceTransformer (`all-MiniLM-L6-v2`)** — embedding model for encoding documents and queries
- **LangChain Groq (`ChatGroq`)** — LLM interface using LLaMA 3.3 70B for routing, answering, and eval scoring
- **LangGraph (`StateGraph`, `MemorySaver`)** — agentic graph framework with persistent conversation memory
- **DuckDuckGo Search (`DDGS`)** — live web search for queries not covered by the PDF knowledge base
- **Streamlit** — browser-based chat UI deployment
- **pyngrok** — ngrok tunnel to expose the Streamlit app publicly from Google Colab

**Deployment:** Streamlit UI — launched via ngrok on Google Colab.


## 0. Setup


In [ ]:
!pip install -q -U langchain-google-genai langgraph chromadb \
             sentence-transformers duckduckgo-search \
             python-dotenv pymupdf streamlit pyngrok google-generativeai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.6/67.6 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 173.7/173.7 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.3/571.3 kB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 34.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 49.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.1/515.1 kB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 50.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/

In [ ]:
!pip install -q langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 4.8 MB/s eta 0:00:00


In [ ]:
from google.colab import userdata
import os
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

In [ ]:
import os, re, time, random
import fitz
from typing import TypedDict, List
from dotenv import load_dotenv
from importlib.metadata import version

from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver

import chromadb
from sentence_transformers import SentenceTransformer

load_dotenv()

groq_key = os.getenv("GROQ_API_KEY", "")
print(f"Groq API Key : {'✅ Loaded' if len(groq_key) > 10 else '❌ Missing'}")
print(f"LangGraph    : {version('langgraph')}")

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
r   = llm.invoke("Say ready in 1 word.")
print(f"LLM          : ✅ {r.content}")

Groq API Key : ✅ Loaded
LangGraph    : 1.1.8
LLM          : ✅ Ready.


## Part 1 — Knowledge Base


In [ ]:
# ============================================================
# CELL 4 — Auto PDF Ingestion + ChromaDB
# ============================================================
# HOW TO ADD MORE PAPERS:
#   1. Upload the PDF to Colab (drag into files panel)
#   2. Add its path to PDF_PAPERS below
#   3. Re-run this cell — KB rebuilds automatically

PDF_PAPERS = [
    "/content/1312.5602v1.pdf",
    "/content/1509.06461v3.pdf",
    "/content/1511.05952v4.pdf",
    "/content/1511.06581v3.pdf",
    "/content/1602.01783v2.pdf",
    "/content/1710.02298v1.pdf"
]
CHUNK_SIZE = 800
EMBED_MODEL = "all-MiniLM-L6-v2"


def _clean(text: str) -> str:
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"[ \t]+",  " ",     text)
    return re.sub(r"\x0c",    "",      text)


def extract_pdf_chunks(pdf_path: str, paper_name: str,
                        chunk_size: int = CHUNK_SIZE) -> list[dict]:
    """Extract and chunk a PDF into ~chunk_size-character documents."""
    print(f"  📄 Reading: {pdf_path}")

    with fitz.open(pdf_path) as doc:
        full_text = _clean("".join(page.get_text() for page in doc))

    # Split by single newline too, not just double newline
    # Academic PDFs often use single \n between paragraphs
    sentences = [s.strip() for s in re.split(r'\n+', full_text) if len(s.strip()) > 40]

    documents, buf, idx = [], "", 1
    for sentence in sentences:
        if len(buf) + len(sentence) < chunk_size:
            buf += " " + sentence
        else:
            if buf.strip():
                documents.append({
                    "id":    f"{paper_name}_{idx:03d}",
                    "paper": paper_name,
                    "topic": f"Section {idx}",
                    "text":  buf.strip()
                })
                idx += 1
            buf = sentence

    if buf.strip():
        documents.append({
            "id":    f"{paper_name}_{idx:03d}",
            "paper": paper_name,
            "topic": f"Section {idx}",
            "text":  buf.strip()
        })

    print(f"  ✅ {len(documents)} chunks from {paper_name}")
    return documents


# ── Ingest all PDFs ──────────────────────────────────────────
print("=" * 55)
print("AUTO PDF INGESTION")
print("=" * 55)

DOCUMENTS: list[dict] = []
for path in PDF_PAPERS:
    name = os.path.splitext(os.path.basename(path))[0]
    try:
        DOCUMENTS.extend(extract_pdf_chunks(path, name))
    except Exception as exc:
        print(f"  ❌ {path}: {exc}")

print(f"\nTotal chunks: {len(DOCUMENTS)}")

# ── Build ChromaDB ───────────────────────────────────────────
print("\nLoading embedding model…")
embedder = SentenceTransformer(EMBED_MODEL)

chroma   = chromadb.Client()
try:    chroma.delete_collection("research_kb")
except: pass
collection = chroma.create_collection("research_kb")

texts = [d["text"]  for d in DOCUMENTS]
ids   = [d["id"]    for d in DOCUMENTS]
metas = [{"topic": d["topic"], "paper": d["paper"]} for d in DOCUMENTS]

collection.add(documents=texts,
               embeddings=embedder.encode(texts, show_progress_bar=True).tolist(),
               ids=ids, metadatas=metas)

print(f"\n✅ Knowledge base: {collection.count()} chunks")
for paper in dict.fromkeys(d["paper"] for d in DOCUMENTS):
    cnt = sum(1 for d in DOCUMENTS if d["paper"] == paper)
    print(f"   📄 {paper} — {cnt} chunks")


AUTO PDF INGESTION
  📄 Reading: /content/1312.5602v1.pdf
  ✅ 44 chunks from 1312.5602v1
  📄 Reading: /content/1509.06461v3.pdf
  ✅ 48 chunks from 1509.06461v3
  📄 Reading: /content/1511.05952v4.pdf
  ✅ 78 chunks from 1511.05952v4
  📄 Reading: /content/1511.06581v3.pdf
  ✅ 50 chunks from 1511.06581v3
  📄 Reading: /content/1602.01783v2.pdf
  ✅ 67 chunks from 1602.01783v2
  📄 Reading: /content/1710.02298v1.pdf
  ✅ 49 chunks from 1710.02298v1

Total chunks: 336

Loading embedding model…


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/11 [00:00<?, ?it/s]


✅ Knowledge base: 336 chunks
   📄 1312.5602v1 — 44 chunks
   📄 1509.06461v3 — 48 chunks
   📄 1511.05952v4 — 78 chunks
   📄 1511.06581v3 — 50 chunks
   📄 1602.01783v2 — 67 chunks
   📄 1710.02298v1 — 49 chunks


In [ ]:
print(f"Total documents in KB : {collection.count()}")
print(f"Total DOCUMENTS list  : {len(DOCUMENTS)}")

Total documents in KB : 336
Total DOCUMENTS list  : 336


In [ ]:
# ============================================================
# CELL 5 — Retrieval Smoke Test
# ============================================================
def retrieve(question: str, n: int = 3) -> tuple[list[str], list[dict]]:
    """Shared retrieval helper used by nodes and tests."""
    q_emb   = embedder.encode([question]).tolist()
    results = collection.query(query_embeddings=q_emb, n_results=n)
    return results["documents"][0], results["metadatas"][0]


test_query = "What algorithm does DQN use for training?"
docs, metas = retrieve(test_query)

print(f"Query: {test_query}")
for i, (doc, meta) in enumerate(zip(docs, metas), 1):
    print(f"\n  [{i}] {meta['paper']} / {meta['topic']}")
    print(f"       {doc[:150]}…")

print("\n✅ Retrieval OK" if docs else "\n❌ Nothing returned — re-run Cell 4")


Query: What algorithm does DQN use for training?

  [1] 1509.06461v3 / Section 8
       where α is a scalar step size and the target Y Q This update resembles stochastic gradient descent, updating the current value Q(St, At; θt) towards a…

  [2] 1602.01783v2 / Section 67
       Breakout, Pong, Q*bert, Space Invaders) for 50 different learning rates and random initializations. All algorithms exhibit some level of robustness to…

  [3] 1509.06461v3 / Section 26
       layers and a fully-connected hidden layer (approximately 1.5M parameters in total). The network takes the last four frames as input and outputs the ac…

✅ Retrieval OK


## Part 2 — State Design


In [ ]:
# ============================================================
# CELL 6 — State Design
# ============================================================
class CapstoneState(TypedDict):
    # Input
    question:     str
    # Memory
    messages:     List[dict]
    # Routing
    route:        str          # "retrieve" | "memory_only" | "tool"
    # RAG
    retrieved:    str
    sources:      List[str]
    # Tool
    tool_result:  str
    # Answer
    answer:       str
    # Quality control
    faithfulness: float
    eval_retries: int
    # Reserved
    paper_filter: str

print("✅ State fields:", list(CapstoneState.__annotations__))


✅ State fields: ['question', 'messages', 'route', 'retrieved', 'sources', 'tool_result', 'answer', 'faithfulness', 'eval_retries', 'paper_filter']


## Part 3 — Node Functions


In [ ]:
# ============================================================
# CELL 7 — Constants + All Node Functions  (rate-limit resilient)
# ============================================================
import time, random

FAITHFULNESS_THRESHOLD = 0.7
MAX_EVAL_RETRIES       = 0
SLIDING_WINDOW         = 6

def _invoke(messages_or_str, *, max_retries: int = 4) -> str:
    if isinstance(messages_or_str, str):
        messages_or_str = [HumanMessage(content=messages_or_str)]
    for attempt in range(max_retries):
        try:
            return llm.invoke(messages_or_str).content
        except Exception as exc:
            msg = str(exc)
            if "429" in msg or "rate_limit" in msg.lower():
                wait = [15, 30, 45, 60][attempt]
                print(f"  ⚠️  Rate limit — retrying in {wait}s (attempt {attempt+1}/{max_retries})")
                time.sleep(wait)
            else:
                raise
    raise RuntimeError("LLM call failed after max retries — quota likely exhausted for today.")

ROUTER_PROMPT = (
    "Route this question. Reply ONE word: retrieve, tool, or memory_only.\n"
    "- retrieve    : paper content, methods, results, architecture\n"
    "- tool        : needs live web info (recent papers, citations)\n"
    "- memory_only : greeting or follow-up, no new retrieval needed"
)

SYSTEM_PROMPT = (
    "You are a Research Paper Q&A assistant.\n"
    "You MUST answer the question using the context provided below.\n"
    "The context contains excerpts from the paper — use them to give a complete answer.\n"
    "Only say 'I don't have that in the knowledge base' if the context is truly empty or completely unrelated.\n"
    "Never refuse to answer when relevant context is present."
)

EVAL_PROMPT = (
    "Rate faithfulness 0.0–1.0. Reply ONE decimal number only.\n"
    "1.0=fully grounded, 0.7=minor inference, 0.5=partial, 0.0=fabricated"
)

def memory_node(state: CapstoneState) -> dict:
    msgs = state.get("messages", []) + [{"role": "user", "content": state["question"]}]
    return {"messages": msgs[-SLIDING_WINDOW:]}

def router_node(state: CapstoneState) -> dict:
    prompt = f"{ROUTER_PROMPT}\n\nQuestion: {state['question']}"
    route  = _invoke(prompt).strip().lower()
    if route not in ("retrieve", "tool", "memory_only"):
        route = "retrieve"
    print(f"  [router] → {route}")
    return {"route": route}

def retrieval_node(state: CapstoneState) -> dict:
    docs, metas = retrieve(state["question"])
    chunks  = [f"[{m['topic']}]\n{d}" for d, m in zip(docs, metas)]
    sources = [m["topic"] for m in metas]
    print(f"  [retrieval] sources: {sources}")
    return {"retrieved": "\n\n---\n\n".join(chunks), "sources": sources}

def skip_retrieval_node(state: CapstoneState) -> dict:
    return {"retrieved": "", "sources": []}

def tool_node(state: CapstoneState) -> dict:
    try:
        from duckduckgo_search import DDGS
        with DDGS() as ddgs:
            results = list(ddgs.text(state["question"] + " research paper", max_results=3))
        if not results:
            return {"tool_result": "No web results found."}
        snippets = [f"[{r.get('title','')}]\n{r.get('body','')[:200]}" for r in results]
        return {"tool_result": "\n\n".join(snippets)}
    except Exception as exc:
        return {"tool_result": f"Web search unavailable: {exc}"}

def answer_node(state: CapstoneState) -> dict:
    parts = []
    if state.get("retrieved"):
        parts.append(f"CONTEXT:\n{state['retrieved'][:1500]}")
    if state.get("tool_result"):
        parts.append(f"WEB:\n{state['tool_result'][:400]}")
    context = "\n\n".join(parts) or "No context."

    history_msgs = state.get("messages", [])[:-1][-1:]
    history = "\n".join(f"{m['role'].upper()}: {m['content'][:80]}" for m in history_msgs)

    retries    = state.get("eval_retries", 0)
    retry_note = f"\n[Retry #{retries}: be specific.]" if retries else ""

    user_prompt = (
        f"{context}\n\n"
        f"History:\n{history}\n\n"
        f"Q: {state['question']}{retry_note}\nA:"
    )
    response = _invoke([SystemMessage(content=SYSTEM_PROMPT),
                         HumanMessage(content=user_prompt)])
    print(f"  [answer] {len(response)} chars")
    return {"answer": response}

def eval_node(state: CapstoneState) -> dict:
    if not state.get("retrieved") and not state.get("tool_result"):
        print("  [eval] skipped — no context")
        return {"faithfulness": 1.0,
                "eval_retries": state.get("eval_retries", 0)}

    ctx    = (state.get("retrieved") or state.get("tool_result", ""))[:300]
    answer = state["answer"][:150]
    prompt = f"{EVAL_PROMPT}\n\nCtx: {ctx}\nAns: {answer}\nScore:"
    raw    = _invoke(prompt).strip()

    try:
        score = max(0.0, min(1.0, float(raw)))
    except ValueError:
        score = 0.5

    retries = state.get("eval_retries", 0) + 1
    gate    = "✅" if score >= FAITHFULNESS_THRESHOLD else "⚠️"
    print(f"  [eval] faithfulness={score:.2f} {gate}  retries={retries - 1}")
    return {"faithfulness": score, "eval_retries": retries}

def save_node(state: CapstoneState) -> dict:
    msgs = state.get("messages", []) + [{"role": "assistant", "content": state["answer"]}]
    return {"messages": msgs}

print("✅ All 7 node functions defined")

✅ All 7 node functions defined


## Part 4 — Graph Assembly


In [ ]:
# ============================================================
# CELL 8 — Graph Assembly
# ============================================================
def route_decision(state: CapstoneState) -> str:
    r = state.get("route", "retrieve")
    return "tool" if r == "tool" else "skip" if r == "memory_only" else "retrieve"


def eval_decision(state: CapstoneState) -> str:
    score   = state.get("faithfulness", 1.0)
    retries = state.get("eval_retries", 0)
    if score < FAITHFULNESS_THRESHOLD and retries < MAX_EVAL_RETRIES:
        print(f"  [eval_decision] RETRY (score={score:.2f})")
        return "answer"
    print(f"  [eval_decision] PASS → save")
    return "save"


graph = StateGraph(CapstoneState)

for name, fn in [("memory",   memory_node),
                  ("router",   router_node),
                  ("retrieve", retrieval_node),
                  ("skip",     skip_retrieval_node),
                  ("tool",     tool_node),
                  ("answer",   answer_node),
                  ("eval",     eval_node),
                  ("save",     save_node)]:
    graph.add_node(name, fn)

graph.set_entry_point("memory")
graph.add_edge("memory",   "router")
graph.add_edge("retrieve", "answer")
graph.add_edge("skip",     "answer")
graph.add_edge("tool",     "answer")
graph.add_edge("answer",   "eval")
graph.add_edge("save",     END)

graph.add_conditional_edges("router", route_decision,
                             {"retrieve": "retrieve", "tool": "tool", "skip": "skip"})
graph.add_conditional_edges("eval",   eval_decision,
                             {"answer": "answer",     "save": "save"})

app = graph.compile(checkpointer=MemorySaver())
print("✅ Graph compiled")
print("   Nodes:", list(app.get_graph().nodes))


✅ Graph compiled
   Nodes: ['__start__', 'memory', 'router', 'retrieve', 'skip', 'tool', 'answer', 'eval', 'save', '__end__']


## Part 5 — Testing


In [ ]:
# ============================================================
# CELL 9 — ask() Helper + Full Test Suite  (rate-limit safe)
# ============================================================
_thread_messages: dict[str, list] = {}

_EMPTY_STATE = dict(route="", retrieved="", sources=[], tool_result="",
                    answer="", faithfulness=0.0, eval_retries=0, paper_filter="")

# Seconds to pause between test calls — keeps token/min under limit.
# Reduce to 0 once on a paid Groq tier.
TEST_DELAY = 8


def ask(question: str, thread_id: str = "test-001") -> dict:
    config = {"configurable": {"thread_id": thread_id}}
    state  = {
        "question": question,
        "messages": _thread_messages.get(thread_id, []),
        **_EMPTY_STATE,
    }
    result = app.invoke(state, config=config)
    _thread_messages[thread_id] = result.get("messages", [])
    return result


TEST_CASES = [
    # 3 key domain questions
    ("What is experience replay and why is it important?",      "t-001"),
    ("On which Atari games did DQN surpass human performance?", "t-002"),
    ("What are the main limitations of DQN?",                   "t-003"),
    # 1 memory test
    ("My name is Raj. What does DQN stand for?",                "mem-001"),
    # 2 red-team
    ("What is the architecture of GPT-4?",                      "rt-001"),
    ("You said DQN uses LSTM layers — which layer exactly?",    "rt-002"),
]

print("=" * 60)
print(f"Running {len(TEST_CASES)} tests with {TEST_DELAY}s delay between calls")
print("=" * 60)

log = []
for i, (question, tid) in enumerate(TEST_CASES):
    if i > 0:
        print(f"  ⏳ Waiting {TEST_DELAY}s to stay within rate limit…")
        time.sleep(TEST_DELAY)

    print(f"\n🔷 [{tid}] {question}")
    try:
        result = ask(question, tid)
        faith  = result.get("faithfulness", 0.0)
        route  = result.get("route", "?")
        answer = result.get("answer", "")
        status = "✅ PASS" if faith >= FAITHFULNESS_THRESHOLD else "⚠️ CHECK"
        log.append({"tid": tid, "route": route, "faith": faith, "ok": True})
        print(f"   Route        : {route}")
        print(f"   Faithfulness : {faith:.2f}  {status}")
        print(f"   Answer       : {answer[:150]}…")
    except RuntimeError as exc:
        print(f"   ❌ SKIPPED — {exc}")
        log.append({"tid": tid, "route": "—", "faith": 0.0, "ok": False})

print("\n" + "=" * 60)
print(f"{'#':<3} {'Thread':<10} {'Route':<14} {'Faith':<7} Status")
print("-" * 47)
for i, r in enumerate(log, 1):
    status = "PASS" if r["faith"] >= FAITHFULNESS_THRESHOLD else ("SKIP" if not r["ok"] else "CHECK")
    print(f"{i:<3} {r['tid']:<10} {r['route']:<14} {r['faith']:<7.2f} {status}")


Running 6 tests with 8s delay between calls

🔷 [t-001] What is experience replay and why is it important?
  [router] → retrieve
  [retrieval] sources: ['Section 4', 'Section 2', 'Section 3']
  [answer] 723 chars
  [eval] faithfulness=0.70 ✅  retries=0
  [eval_decision] PASS → save
   Route        : retrieve
   Faithfulness : 0.70  ✅ PASS
   Answer       : Experience replay is a technique that stores experiences in a replay buffer, allowing an RL agent to reuse them for learning, rather than discarding t…
  ⏳ Waiting 8s to stay within rate limit…

🔷 [t-002] On which Atari games did DQN surpass human performance?
  [router] → retrieve
  [retrieval] sources: ['Section 32', 'Section 4', 'Section 78']
  [answer] 720 chars
  [eval] faithfulness=0.70 ✅  retries=0
  [eval_decision] PASS → save
   Route        : retrieve
   Faithfulness : 0.70  ✅ PASS
   Answer       : According to the context, DQN surpassed human performance on some Atari games, but the specific games are not mentioned in the 

## Part 6 — RAGAS Baseline Evaluation


In [ ]:
# ============================================================
# CELL 10 — Evaluation (rate-limit safe)
# ============================================================
RAGAS_QUESTIONS = [
    {
        "question":    "What is experience replay?",
        "ground_truth": (
            "Experience replay stores transitions (s, a, r, s') in a replay memory D. "
            "Random minibatches are sampled from D to perform Q-learning updates, "
            "breaking correlations between consecutive samples."
        ),
    },
    {
        "question":    "What neural network architecture does DQN use?",
        "ground_truth": (
            "Input: 84×84×4. Conv1: 16 filters 8×8 stride 4 + ReLU. "
            "Conv2: 32 filters 4×4 stride 2 + ReLU. FC: 256 units + ReLU. "
            "Output: one node per valid action."
        ),
    },
    {
        "question":    "On which games did DQN surpass human performance?",
        "ground_truth": "Breakout, Enduro, and Pong.",
    },
    {
        "question":    "What are the limitations of DQN?",
        "ground_truth": (
            "Uniform replay sampling without prioritisation; reward clipping; "
            "poor performance on long-horizon games; partial observability."
        ),
    },
    {
        "question":    "What preprocessing is applied to Atari frames?",
        "ground_truth": (
            "210×160 RGB → grayscale → 110×84 → crop 84×84. "
            "Last 4 frames stacked → 84×84×4 input."
        ),
    },
]

EVAL_DELAY = 8  # seconds between scoring calls

def _score_llm(prompt: str) -> float:
    try:
        return max(0.0, min(1.0, float(_invoke(prompt).strip().split()[0])))
    except Exception:
        return 0.5


# Build eval dataset
eval_dataset = []
print("Running agent for evaluation…")
for i, rq in enumerate(RAGAS_QUESTIONS):
    if i > 0:
        print(f"  ⏳ Waiting {EVAL_DELAY}s…")
        time.sleep(EVAL_DELAY)
    docs, _ = retrieve(rq["question"])
    result  = ask(rq["question"], thread_id=f"ragas-{i}")
    eval_dataset.append({
        "question":     rq["question"],
        "answer":       result.get("answer", ""),
        "contexts":     docs,
        "ground_truth": rq["ground_truth"],
    })
    print(f"  ✓ {rq['question'][:60]}")

print(f"\n✅ Eval dataset: {len(eval_dataset)} rows")
print("=" * 50)
print("BASELINE EVALUATION")
print("=" * 50)

faith_scores, rel_scores = [], []
for i, row in enumerate(eval_dataset):
    if i > 0:
        print(f"  ⏳ Waiting {EVAL_DELAY}s…")
        time.sleep(EVAL_DELAY)

    f = _score_llm(
        f"Rate faithfulness 0.0-1.0. ONE number only.\n"
        f"Context: {row['contexts'][0][:200]}\nAnswer: {row['answer'][:150]}"
    )
    faith_scores.append(f)
    print(f"Q: {row['question'][:55]}")
    print(f"   Faithfulness : {f:.2f}")

print("\n" + "=" * 50)
print(f"AVG Faithfulness : {sum(faith_scores)/len(faith_scores):.3f}")
print("\n⚠️  Record this in your written summary.")

Running agent for evaluation…
  [router] → retrieve
  [retrieval] sources: ['Section 2', 'Section 4', 'Section 3']
  [answer] 488 chars
  [eval] faithfulness=0.70 ✅  retries=0
  [eval_decision] PASS → save
  ✓ What is experience replay?
  ⏳ Waiting 8s…
  [router] → retrieve
  [retrieval] sources: ['Section 8', 'Section 26', 'Section 2']
  [answer] 377 chars
  [eval] faithfulness=0.90 ✅  retries=0
  [eval_decision] PASS → save
  ✓ What neural network architecture does DQN use?
  ⏳ Waiting 8s…
  [router] → retrieve
  [retrieval] sources: ['Section 32', 'Section 78', 'Section 28']
  [answer] 511 chars
  [eval] faithfulness=0.70 ✅  retries=0
  [eval_decision] PASS → save
  ✓ On which games did DQN surpass human performance?
  ⏳ Waiting 8s…
  [router] → retrieve
  [retrieval] sources: ['Section 26', 'Section 29', 'Section 47']
  [answer] 613 chars
  [eval] faithfulness=0.80 ✅  retries=0
  [eval_decision] PASS → save
  ✓ What are the limitations of DQN?
  ⏳ Waiting 8s…
  [router] → retrieve


## Part 7 — Deployment


In [ ]:
streamlit_code = """
import streamlit as st
import uuid, os, re
import fitz
import chromadb
from dotenv import load_dotenv
from typing import TypedDict, List
from sentence_transformers import SentenceTransformer
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver

load_dotenv()

st.set_page_config(page_title="Research Paper Q&A", layout="centered")
st.title(" Research Paper Q&A Agent")
st.caption("Ask questions about Deep Reinforcement Learning research.")

def _clean(text):
    text = re.sub(r'\\n{3,}', '\\n\\n', text)
    return re.sub(r'[ \\t]+', ' ', text)

def _chunk_pdf(pdf_path, paper_name, chunk_size=800):
    with fitz.open(pdf_path) as doc:
        raw = _clean(''.join(p.get_text() for p in doc))
    lines = [s.strip() for s in re.split(r'\\n+', raw) if len(s.strip()) > 40]
    docs, buf, idx = [], '', 1
    for line in lines:
        if len(buf) + len(line) < chunk_size:
            buf += ' ' + line
        else:
            if buf.strip():
                docs.append({'id': f'{paper_name}_{idx:03d}', 'paper': paper_name,
                              'topic': f'Section {idx}', 'text': buf.strip()})
                idx += 1
            buf = line
    if buf.strip():
        docs.append({'id': f'{paper_name}_{idx:03d}', 'paper': paper_name,
                     'topic': f'Section {idx}', 'text': buf.strip()})
    return docs

@st.cache_resource
def load_agent():
    llm      = ChatGroq(model='llama-3.3-70b-versatile', temperature=0)
    embedder = SentenceTransformer('all-MiniLM-L6-v2')

    PDF_PAPERS = [
        '/content/1312.5602v1.pdf',    # DQN original
        '/content/1509.06461.pdf',     # Double DQN
        '/content/1511.06581.pdf',     # Dueling DQN
        '/content/1511.05952.pdf',     # Prioritized Replay
        '/content/1710.02298.pdf',     # Rainbow
        '/content/1602.01783.pdf',     # A3C
    ]

    cli = chromadb.Client()
    try:    cli.delete_collection('research_kb')
    except: pass
    col = cli.create_collection('research_kb')

    all_docs = []
    for path in PDF_PAPERS:
        name = os.path.splitext(os.path.basename(path))[0]
        try:
            all_docs.extend(_chunk_pdf(path, name))
        except Exception as e:
            print(f'Warning: could not load {path}: {e}')

    texts = [d['text'] for d in all_docs]
    col.add(
        documents=texts,
        embeddings=embedder.encode(texts).tolist(),
        ids=[d['id'] for d in all_docs],
        metadatas=[{'topic': d['topic'], 'paper': d['paper']} for d in all_docs]
    )

    THRESHOLD, MAX_RETRIES = 0.7, 0

    class S(TypedDict):
        question: str; messages: List[dict]; route: str
        retrieved: str; sources: List[str]; tool_result: str
        answer: str; faithfulness: float; eval_retries: int; paper_filter: str

    SYSTEM_PROMPT = (
        'You are a Research Paper Q&A assistant.\\n'
        'You MUST answer the question using the context provided below.\\n'
        'The context contains excerpts from the paper — use them to give a complete answer.\\n'
        'Only say I do not have that in the knowledge base if the context is truly empty or completely unrelated.\\n'
        'Never refuse to answer when relevant context is present.'
    )

    def memory(s):
        msgs = s.get('messages', []) + [{'role': 'user', 'content': s['question']}]
        return {'messages': msgs[-6:]}

    def router(s):
        r = llm.invoke(
            f"Route: retrieve / tool / memory_only.\\nQuestion: {s['question']}\\nOne word:"
        ).content.strip().lower()
        if 'memory' in r: r = 'memory_only'
        elif 'tool' in r: r = 'tool'
        else: r = 'retrieve'
        return {'route': r}

    def retrieval(s):
        q   = embedder.encode([s['question']]).tolist()
        res = col.query(query_embeddings=q, n_results=3)
        chunks  = [f"[{m['topic']}]\\n{d}" for d, m in zip(res['documents'][0], res['metadatas'][0])]
        sources = [m['topic'] for m in res['metadatas'][0]]
        return {'retrieved': '\\n\\n---\\n\\n'.join(chunks), 'sources': sources}

    def skip(s):
        return {'retrieved': '', 'sources': []}

    def tool(s):
        try:
            from duckduckgo_search import DDGS
            with DDGS() as ddgs:
                hits = list(ddgs.text(s['question'] + ' research paper', max_results=3))
            return {'tool_result': '\\n\\n'.join(f"[{r['title']}]\\n{r['body'][:200]}" for r in hits)}
        except Exception as exc:
            return {'tool_result': f'Search unavailable: {exc}'}

    def answer(s):
        parts = []
        if s.get('retrieved'):
            parts.append(f"CONTEXT:\\n{s['retrieved'][:1500]}")
        if s.get('tool_result'):
            parts.append(f"WEB SEARCH:\\n{s['tool_result'][:300]}")
        ctx    = '\\n\\n'.join(parts) or 'No context.'
        system = SYSTEM_PROMPT + f'\\n\\n{ctx}'
        if s.get('eval_retries', 0) > 0:
            system += '\\n\\nIMPORTANT: Use ONLY explicit context.'
        msgs = [SystemMessage(content=system)]
        for m in s.get('messages', [])[:-1]:
            cls = HumanMessage if m['role'] == 'user' else AIMessage
            msgs.append(cls(content=m['content']))
        msgs.append(HumanMessage(content=s['question']))
        return {'answer': llm.invoke(msgs).content}

    def eval_(s):
        ctx = s.get('retrieved', '')[:500]
        if not ctx:
            return {'faithfulness': 1.0, 'eval_retries': s.get('eval_retries', 0) + 1}
        try:
            score = max(0.0, min(1.0, float(
                llm.invoke(
                    f"Rate faithfulness 0.0-1.0. ONE number only.\\nContext: {ctx}\\nAnswer: {s.get('answer','')[:300]}"
                ).content.strip().split()[0])))
        except Exception:
            score = 0.5
        return {'faithfulness': score, 'eval_retries': s.get('eval_retries', 0) + 1}

    def save(s):
        return {'messages': s.get('messages', []) + [{'role': 'assistant', 'content': s['answer']}]}

    def route_dec(s):
        r = s.get('route', 'retrieve')
        return 'tool' if r == 'tool' else 'skip' if r == 'memory_only' else 'retrieve'

    def eval_dec(s):
        return ('save'
                if s.get('faithfulness', 1.0) >= THRESHOLD or s.get('eval_retries', 0) >= MAX_RETRIES
                else 'answer')

    g = StateGraph(S)
    for name, fn in [('memory', memory), ('router', router), ('retrieve', retrieval),
                     ('skip', skip), ('tool', tool), ('answer', answer),
                     ('eval', eval_), ('save', save)]:
        g.add_node(name, fn)
    g.set_entry_point('memory')
    g.add_edge('memory', 'router')
    g.add_conditional_edges('router', route_dec,
                             {'retrieve': 'retrieve', 'skip': 'skip', 'tool': 'tool'})
    for n in ('retrieve', 'skip', 'tool'):
        g.add_edge(n, 'answer')
    g.add_edge('answer', 'eval')
    g.add_conditional_edges('eval', eval_dec, {'answer': 'answer', 'save': 'save'})
    g.add_edge('save', END)

    agent_app = g.compile(checkpointer=MemorySaver())
    return agent_app, col.count()

with st.spinner('Loading knowledge base...'):
    agent_app, doc_count = load_agent()

for key, default in [('messages', []), ('thread_id', str(uuid.uuid4())[:8])]:
    if key not in st.session_state:
        st.session_state[key] = default



for msg in st.session_state.messages:
    with st.chat_message(msg['role']):
        st.write(msg['content'])

if prompt := st.chat_input('Ask about the paper...'):
    with st.chat_message('user'):
        st.write(prompt)
    st.session_state.messages.append({'role': 'user', 'content': prompt})

    with st.chat_message('assistant'):
        with st.spinner('Thinking...'):
            result = agent_app.invoke(
                {'question': prompt,
                 'messages': st.session_state.messages[:-1],
                 'route': '', 'retrieved': '', 'sources': [], 'tool_result': '',
                 'answer': '', 'faithfulness': 0.0, 'eval_retries': 0, 'paper_filter': ''},
                config={'configurable': {'thread_id': st.session_state.thread_id}},
            )
            answer = result.get('answer', 'Sorry, could not generate an answer.')
        st.write(answer)
        faith = result.get('faithfulness', 0.0)
        if faith:
            st.caption(f"Route: {result.get('route','?')} | Faithfulness: {faith:.2f}")

    st.session_state.messages.append({'role': 'assistant', 'content': answer})
"""

with open("capstone_streamlit.py", "w", encoding="utf-8") as fh:
    fh.write(streamlit_code.lstrip())

print("✅ capstone_streamlit.py written successfully")

✅ capstone_streamlit.py written successfully


## Part 8 — Written Summary


## Summary

**Name:** Nishant Sahoo

**Domain:** Research Paper Q&A — Deep Reinforcement Learning

**User:** PhD students and AI researchers.

**What the agent does:** The agent answers natural language questions about six foundational Deep Reinforcement Learning research papers. It ingests and chunks PDF content into a ChromaDB vector store (336 chunks total), retrieves the top-3 relevant chunks per query using `all-MiniLM-L6-v2` sentence-transformer embeddings, and generates grounded answers via Groq-hosted LLaMA 3.3 70B. For queries needing live or recent information it routes to DuckDuckGo web search. A self-reflection eval node scores faithfulness (0.0–1.0) and gates low-quality answers before saving to memory.

**Knowledge base:** 6 research PDFs ingested and chunked into 800-character segments using PyMuPDF. Total: **336 chunks**.
- `1312.5602v1` — DQN original — 44 chunks
- `1509.06461v3` — Double DQN — 48 chunks
- `1511.05952v4` — Prioritized Experience Replay — 78 chunks
- `1511.06581v3` — Dueling DQN — 50 chunks
- `1602.01783v2` — A3C — 67 chunks
- `1710.02298v1` — Rainbow — 49 chunks

**Tools used:**
- **PyMuPDF (`fitz`)** — extracts and cleans raw text from PDF pages; splits into 800-character chunks for ChromaDB ingestion
- **ChromaDB** — in-memory vector store; stores 336 chunks with embeddings and metadata; queried with `n_results=3` per question
- **SentenceTransformer (`all-MiniLM-L6-v2`)** — encodes documents at ingestion and queries at retrieval into 384-dim vectors
- **LangChain Groq (`ChatGroq`, LLaMA 3.3 70B)** — router decisions, answer generation, and faithfulness eval scoring
- **LangGraph (`StateGraph`, `MemorySaver`)** — orchestrates 8 nodes (memory → router → retrieve/skip/tool → answer → eval → save) with persistent thread-based memory
- **DuckDuckGo Search (`DDGS`)** — live web search invoked by `tool_node` when the router routes to `tool`; returns top-3 snippets
- **Streamlit** — browser-based chat UI with `@st.cache_resource` for model loading and `st.session_state` for message history
- **pyngrok** — exposes the Streamlit app on port 8501 via a public ngrok HTTPS tunnel from Google Colab

**RAGAS baseline scores** (manual LLM faithfulness — RAGAS library not installed):
- What is experience replay? → **0.80**
- What neural network architecture does DQN use? → **0.80**
- On which games did DQN surpass human performance? → **0.80**
- What are the limitations of DQN? → **0.80**
- What preprocessing is applied to Atari frames? → **0.00**
- **Average Faithfulness: 0.640**

**Test results:** 6 / 6 tests run — 5 PASS, 1 CHECK.

| # | Thread | Question | Route | Faithfulness | Status |
|---|--------|----------|-------|-------------|--------|
| 1 | t-001 | What is experience replay and why is it important? | retrieve | 0.70 | ✅ PASS |
| 2 | t-002 | On which Atari games did DQN surpass human performance? | retrieve | 0.70 | ✅ PASS |
| 3 | t-003 | What are the main limitations of DQN? | retrieve | 0.80 | ✅ PASS |
| 4 | mem-001 | My name is Raj. What does DQN stand for? | memory_only | 1.00 | ✅ PASS |
| 5 | rt-001 | What is the architecture of GPT-4? *(red-team)* | retrieve | 0.00 | ⚠️ CHECK |
| 6 | rt-002 | You said DQN uses LSTM layers — which layer exactly? *(red-team)* | memory_only | 1.00 | ✅ PASS |

**Red-team:** 2 / 2 — both adversarial prompts correctly responded with `I don't have that in the knowledge base.`

**One thing I would improve with more time:** Replace fixed 800-character chunking with semantic chunking — splitting on PDF section headers and paragraph boundaries detected via font-size heuristics in PyMuPDF. This would fix the 0.00 faithfulness score on the Atari preprocessing question, where the answer was split mid-context across two chunks and neither chunk alone was sufficient.

**Most surprising thing I learned:** The router prompt wording has a large effect on routing accuracy. The `mem-001` test was routed to `memory_only` and answered `I don't have that in the knowledge base` — the router treated the name-introduction as a greeting and skipped retrieval entirely, missing the DQN question. A clearer router prompt distinguishing conversational openers from hybrid questions would fix this.


In [ ]:
# ============================================================
# CELL 12 — Launch Streamlit via ngrok
# ============================================================
from pyngrok import ngrok
ngrok.kill()
print("✅ Old tunnels cleared")

import subprocess, time
from pyngrok import ngrok

ngrok.set_auth_token("2kTZ3HLdbEnXT2wEbCqJjZtiC7r_6SnXX2sd2EgeAjVHVHhbj")   # ← paste your token

subprocess.Popen([
    "streamlit", "run", "capstone_streamlit.py",
    "--server.port", "8501",
    "--server.headless", "true",
])
time.sleep(5)

url = ngrok.connect(8501)
print(f"\n🚀 Live at: {url}")


✅ Old tunnels cleared

🚀 Live at: NgrokTunnel: "https://72a6-34-56-194-46.ngrok-free.app" -> "http://localhost:8501"
